In [1]:
import sys, json; sys.path.insert(0, '..')
import pandas as pd, numpy as np
from IPython.display import Image, display
from src import preprocessing as pp, feature_engineering as fe, pipeline as P
from src.config import *
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
def show(name, w=800): display(Image(filename=str(FIG / name), width=w))

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 02 Data understanding
Structure, missingness, duplicates, join key and cardinality.

In [2]:
gen, prov = pp.load_raw()
ins = pd.concat([pp.inspect_table(gen,'General'), pp.inspect_table(prov,'Provider')]); ins

,file,dtype,missing_%,n_unique,example
CMS Certification Number (CCN),General,str,0.0,1222,013025
Provider Name,General,str,0.0,1207,ENCOMPASS HEALTH LAKESHORE REHABILITATION HOSP...
Address Line 1,General,str,0.0,1220,3800 RIDGEWAY DRIVE
Address Line 2,General,str,0.0,68,-
City/Town,General,str,0.0,830,BIRMINGHAM
State,General,str,0.0,52,AL
ZIP Code,General,str,0.0,1126,35209
County/Parish,General,str,0.0,537,Jefferson
Telephone Number,General,str,0.0,1136,(205) 868-2000
CMS Region,General,str,0.0,10,4


In [3]:
print('Duplicate rows  general/provider:', gen.duplicated().sum(), prov.duplicated().sum())
print('Unique CCN        general/provider:', gen[CCN].nunique(), prov[CCN].nunique())
print('Provider CCN all in General:', prov[CCN].isin(gen[CCN]).all(), '| General CCN all in Provider:', gen[CCN].isin(prov[CCN]).all())
print('Rows per CCN in Provider:', prov.groupby(CCN).size().unique())
print('Duplicate (CCN, Measure Code):', prov.duplicated([CCN,'Measure Code']).sum())
print('=> General:Provider is one-to-many (1 : 65).')

Duplicate rows  general/provider: 0 0
Unique CCN        general/provider: 1222 1222


Provider CCN all in General: True | General CCN all in Provider: True
Rows per CCN in Provider: [65]
Duplicate (CCN, Measure Code): 0
=> General:Provider is one-to-many (1 : 65).


In [4]:
prov['Measure Code'].str.extract(r'^(I_\d{3}_\d{2})')[0].value_counts().sort_index()

0
I_006_01    8554
I_011_05    3666
I_012_06    3666
I_013_01    3666
I_015_01    8554
I_016_01    3666
I_017_01    8554
I_018_01    8554
I_019_02    8554
I_020_01    2444
I_021_01    3666
I_022_01    4888
I_024_01    3666
I_025_02    3666
I_026_01    3666
Name: count, dtype: int64

In [5]:
prov.groupby('Measure Date Range').size().head(3); prov[['Start Date','End Date']].drop_duplicates()

,Start Date,End Date
0,10/01/2024,09/30/2025
7,01/01/2025,12/31/2025
23,10/01/2024,03/31/2025
26,10/01/2023,09/30/2025
47,10/01/2022,09/30/2024
